# Patient cohorts

This tutorial uses a COVID-19 cohort in which some patients were sampled on more than one day. The data has three
levels: patients with their clinical covariates, samples with their collection day, and cells. We split it into these
levels, select at each level, move information between levels, compute sample representations with patpy, and set up
cross validation in which all samples of a patient stay on the same side of a split.

We use the PBMC cohort of [Stephenson et al. (2021)](https://doi.org/10.1038/s41591-021-01329-2), subsampled to
about 500 cells per sample for the pertpy example data. It covers 119 patients with disease status, severity,
outcome, smoking status, sex and the site that collected the blood.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

from hidata import HiData, read_hd

adata = sc.read(
    "data/stephenson_2021_subsampled.h5ad",
    backup_url="https://scverse-exampledata.s3.eu-west-1.amazonaws.com/pertpy/stephenson_2021_subsampled.h5ad",
)
adata

Ten patients were sampled on more than one day. A sample is a patient on a collection day, so we give every sample
its own identifier. A coarse cell type annotation keeps the plots readable.

In [ ]:
adata.obs["sample_id"] = (adata.obs["patient_id"].astype(str) + "_" + adata.obs["Collection_Day"].astype(str)).astype(
    "category"
)

rules = [
    ("mono", "Monocyte"),
    ("Mono", "Monocyte"),
    ("Plasma", "Plasma"),
    ("CD4", "CD4 T"),
    ("Treg", "CD4 T"),
    ("CD8", "CD8 T"),
    ("NKT", "Other lymphoid"),
    ("NK", "NK"),
    ("B_", "B"),
    ("DC", "DC"),
    ("gdT", "Other lymphoid"),
    ("MAIT", "Other lymphoid"),
    ("ILC", "Other lymphoid"),
]
categories = adata.obs["author_cell_type"].cat.categories
coarse = {name: next((value for key, value in rules if key in name), "Other") for name in categories}
adata.obs["cell_type_coarse"] = adata.obs["author_cell_type"].map(coarse).astype("category")
adata.obs["cell_type_coarse"].value_counts()

## Three levels in one object

`HiData.from_anndata` checks every column of `adata.obs`. With the patient as the group and the sample as the
subgroup, columns that take a single value within every patient move to the patient table `G`, columns that are
constant within samples but differ between the samples of a patient move to the sample table `S`, and everything
that varies between cells stays in `C`.

In [ ]:
hd = HiData.from_anndata(adata, group_id="patient_id", subgroup_id="sample_id")
hd

In [ ]:
hd.G.head()

In [ ]:
hd.S.head()

In [ ]:
list(hd.C.obs.columns)

Patient covariates are stored once per patient in `G`. Here we add an ordered severity scale. Cells are stored
grouped by sample and samples grouped by patient, so the cells of a unit form a contiguous block of `C`.

In [ ]:
severity = ["Healthy", "Asymptomatic", "Mild", "Moderate", "Severe", "Critical"]
hd.G["severity"] = pd.Categorical(hd.G["Status_on_day_collection_summary"], categories=severity, ordered=True)
pd.DataFrame({"patients": hd.G["severity"].value_counts(sort=False)})

In [ ]:
hd.n_subgroups_per_group.value_counts()

## Selecting at every level

A selection at one level is passed on to the others. Selecting patients selects their samples and cells, selecting
samples drops patients without samples, and selecting cells drops samples and patients without cells.

In [ ]:
def sizes(data):
    return {"patients": data.n_groups, "samples": data.n_subgroups, "cells": data.n_cells}


pd.DataFrame(
    {
        "all": sizes(hd),
        "COVID-19 at Ncl": sizes(hd.filter_groups("Status == 'Covid' and Site == 'Ncl'")),
        "first visit": sizes(hd.filter_subgroups("Collection_Day == 'D0'")),
        "monocytes": sizes(hd.filter_cells("cell_type_coarse == 'Monocyte'")),
    }
)

Quality filters are usually defined per sample. Here we keep samples with at least 20 cells of every major cell
type, the requirement of cell type specific pseudobulk and composition analyses. Patients whose samples all fail
leave the cohort with them.

In [ ]:
major_types = ["B", "CD4 T", "CD8 T", "Monocyte", "NK"]
qc = hd.filter_groups_by_cells(20, groupby="cell_type_coarse", groups=major_types, level="subgroup")
sizes(qc)

## Information from any level

`get_df` collects variables from any level into one table at the level you ask for. Patient and sample covariates
are repeated for their cells, and cell values are averaged for their samples or patients. Keys can be columns, genes
or matrices in `obsm`, which expand into one column per dimension.

In [ ]:
hd.get_df(["severity", "Collection_Day", "cell_type_coarse", "CD14"], level="cell").head()

In [ ]:
hd.get_df(["severity", "Site", "CD14", "X_scVI"], level="subgroup").iloc[:5, :7]

## Composition per patient and per sample

`composition` counts cell groups per unit and stores the result in `obsm` of that level, where it follows every
later selection. Pooled over the samples of a patient, it shows how blood composition shifts with severity.

In [ ]:
fractions = hd.composition("cell_type_coarse")
mean_by_severity = fractions.groupby(hd.G["severity"], observed=True).mean()
mean_by_severity.plot(kind="bar", stacked=True, figsize=(7, 4), width=0.8, colormap="tab20")
plt.ylabel("Mean fraction of cells")
plt.legend(loc=(1.02, 0), title="Cell type")
plt.show()

Critically ill patients carry more B cells and fewer CD8 T cells in their blood, moderate cases have the most
monocytes, and plasma cells expand in severe disease.

With the sample level, compositions can also be compared between two visits of the same patient. We take the
patients with two samples and compare the centered log ratio of their compositions, the usual transform for
compositional data, between the earlier and the later sample.

In [ ]:
repeated = hd.filter_groups(hd.n_subgroups_per_group > 1)
clr = repeated.composition("cell_type_coarse", clr=True, level="subgroup")
visits = repeated.S.assign(day=repeated.S["Collection_Day"].astype(str).str[1:].astype(int))
earlier = visits.sort_values("day").groupby("patient_id", observed=True).head(1).index
later = visits.index.difference(earlier)
change = (
    clr.loc[later].set_axis(visits.loc[later, "patient_id"])
    - clr.loc[earlier].set_axis(visits.loc[earlier, "patient_id"])
).sort_index()
change.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
change.boxplot(ax=ax, grid=False)
ax.axhline(0, color="grey", lw=0.5)
ax.set_ylabel("Change of CLR, later minus earlier sample")
plt.xticks(rotation=45, ha="right")
plt.show()

Between the earlier and the later sample, plasma cells fall in nine of ten patients while CD8 T and NK cells rise,
as expected when the acute antibody response resolves and lymphocytes recover. One patient, CV0201, moves the other
way.

## Sample representations with patpy

[patpy](https://patpy.readthedocs.io) computes representations of samples from their cells. Given a HiData with a
subgroup level, its methods represent the samples, read covariates from the sample and patient tables, and write
their results back into the subgroup level: the representation and its 2D embedding into `obsm`, the sample
distances into `obsp`. Here we use the centered log ratio transformed composition, a baseline that performed well in
the patpy benchmark, and the scVI latent space averaged over the cells of every sample.

In [ ]:
import patpy

composition = patpy.tl.CellGroupComposition(sample_key="sample_id", cell_group_key="cell_type_coarse", apply_clr=True)
composition.prepare_anndata(hd)
composition.calculate_distance_matrix()
composition.embed("MDS")
composition.to_donordata("composition_clr")

hd.aggregate(obsm="X_scVI", key_added="scVI_mean", level="subgroup")
hd.levels["subgroup"]

`pseudobulk` turns a level into an AnnData with one averaged expression profile per unit. It carries the sample
table, the patient covariates and all sample-aligned matrices, so scanpy plotting works on it directly.

In [ ]:
samples = hd.pseudobulk(func="mean", level="subgroup")
sc.pl.embedding(samples, basis="X_mds_composition_clr", color=["severity", "Site", "CD14"], ncols=3, frameon=False)

The composition embedding separates the collection sites more clearly than the severity groups, and the average CD14
expression of a sample differs between sites as well, which points to differences in sample processing.

Distances follow selections of samples or patients. The distances among COVID-19 samples equal the corresponding
block of the full matrix.

In [ ]:
covid = hd.filter_groups("Status == 'Covid'")
full = hd.levels["subgroup"].obsp["composition_clr_distances"]
block = covid.levels["subgroup"].obsp["composition_clr_distances"]
positions = hd.S.index.get_indexer(covid.S.index)
np.allclose(block, full[np.ix_(positions, positions)])

patpy scores a representation by how well a nearest neighbor predictor recovers sample covariates from the stored
distances. Severity should be predictable, the collection site should not. Covariates of the patient table are
repeated for every sample of the patient.

In [ ]:
from patpy.tl.evaluation import knn_prediction_score

known = hd.filter_groups(hd.G["severity"].notna())
known.G["severity_rank"] = known.G["severity"].cat.codes.astype(float)
schema = {
    "relevant": {"severity_rank": "ranking", "Status": "classification"},
    "technical": {"Site": "classification"},
}
knn_prediction_score(known.flatten().levels["group"], schema, representations=["composition_clr"], n_neighbors=5)[
    ["covariate", "metric", "score"]
]

Severity is recovered with a rank correlation of 0.42. Technical scores are reported as one minus the prediction
score, so that higher means less batch effect, and the 0.42 for the site therefore means that the site is predicted
with a calibrated F1 of 0.58, better than the COVID-19 status. This representation should be corrected for the
collection site before its structure is read biologically.

## Patients, not samples, in train and test

Two samples of the same patient are more alike than samples of different patients, so they should not be split
between training and test data. `kfold` and `train_test_split` draw folds
over patients, so all samples and cells of a patient stay together. Here a logistic regression predicts COVID-19 from
the averaged scVI profile of every sample, with balanced class weights because healthy donors are a minority.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cohort = hd.filter_groups("Status in ['Covid', 'Healthy']")
cohort.G["Status"] = cohort.G["Status"].cat.remove_unused_categories()

scores = []
for train, test in cohort.kfold(5, stratify="Status", random_state=0):
    assert not set(train.G.index) & set(test.G.index)
    features = {name: part.get_df("scVI_mean", level="subgroup") for name, part in (("train", train), ("test", test))}
    labels = {
        name: part.get_df("Status", level="subgroup")["Status"] for name, part in (("train", train), ("test", test))
    }
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    model.fit(features["train"], labels["train"])
    scores.append(balanced_accuracy_score(labels["test"], model.predict(features["test"])))
print("balanced accuracy per fold:", np.round(scores, 2), "mean:", round(float(np.mean(scores)), 2))

The spread between folds comes from the small number of healthy patients in every test fold.

Models trained on single cells face the same imbalance one level down. Patients with many cells dominate the loss,
and so does the larger class. `cell_weights` gives every patient the same total weight and, with `by`, every class
of a patient covariate as well.

In [ ]:
weights = cohort.cell_weights(by="Status")
cell_status = cohort.get_df("Status", level="cell")["Status"]
pd.DataFrame(
    {"cells": cell_status.value_counts(), "total weight": weights.groupby(cell_status, observed=True).sum().round(0)}
)

## Bags of cells for multiple instance learning

Models that look at every cell of a sample, such as attention based multiple instance learning, need one bag of
cells per sample. `MILDataset` reads the bags as contiguous slices of `C`, subsamples or pads them to a fixed size,
takes labels from the sample or patient table, and works with a standard PyTorch `DataLoader`. The supervised methods
of patpy accept a HiData in the same way.

In [ ]:
from torch.utils.data import DataLoader

from hidata.ml import MILDataset, mil_collate_fn

dataset = MILDataset(
    cohort, level="subgroup", cell_layer="X_scVI", group_labels_key="Status", n_cells=256, random_state=0
)
loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=mil_collate_fn)
batch = next(iter(loader))
batch["cell_x"].shape, batch["cell_mask"].shape, dataset.encodings["Status"]

## Pseudobulk profiles for differential expression

With `groupby`, `pseudobulk` returns one profile per sample and cell type. Its `obs` holds the sample table together
with the patient covariates, so a design such as `~ sex + Site + severity` can be written directly, and
`n_cells` and `total_counts` record how much data every profile rests on. This dataset only ships log normalized
expression, so we average here; with raw counts in a layer use `layer="counts"` and `func="sum"`.

In [ ]:
profiles = qc.pseudobulk(groupby="cell_type_coarse", func="mean", min_cells=20, level="subgroup")
profiles.obs[["patient_id", "Collection_Day", "cell_type_coarse", "n_cells", "sex", "Site", "severity"]].head()

## Adding or removing the patient level

Cohorts are often shared with one row per sample. `nest` adds the patient level to such an object, moving the
columns that are constant within patients into the patient table. `flatten` goes the other way and makes the samples
the units, with patient covariates repeated for every sample.

In [ ]:
by_sample = HiData.from_anndata(adata, group_id="sample_id")
nested = by_sample.nest("patient_id")
flat = nested.flatten()
for name, data in [("by sample", by_sample), ("nested", nested), ("flattened", flat)]:
    print(f"{name:10} levels {data.level_names}, group_id={data.group_id!r}, subgroup_id={data.subgroup_id!r}")

## Cells in the context of their patient

`to_anndata` attaches sample and patient covariates to every cell and shares all matrices with `C`, for example for
cell level plots with tools that work on a single AnnData.

In [ ]:
cells = hd.to_anndata(columns=["severity", "Collection_Day"])
sc.pl.umap(cells, color=["cell_type_coarse", "severity"], frameon=False, ncols=2)

## Saving

Patients, samples, cells and every matrix aligned to them are written to a single file.

In [ ]:
hd.write_hd("data/stephenson.hd.h5")
read_hd("data/stephenson.hd.h5")